# TF-IDF Class Activity

## 0. Setup — install & download NLTK data

In [1]:
!pip install -q nltk scikit-learn numpy

import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("NLTK ready:", nltk.__version__)

NLTK ready: 3.9.1


## 1. Load the data
The 10 titles and documents from `data/documents.py`, embedded here.

In [2]:
TITLES = [
    "AI in Healthcare",
    "Climate Change",
    "Basketball Strategy",
    "Italian Pasta",
    "Space Exploration",
    "Cybersecurity",
    "Music Theory",
    "Gardening Vegetables",
    "Economics Inflation",
    "Public Health",
]

DOCUMENTS = [
    """Artificial intelligence is transforming healthcare by helping doctors analyze medical images, predict disease risk, and personalize treatments. Machine learning models can scan thousands of patient records to find patterns that humans might miss. Hospitals use AI to reduce waiting times and improve diagnosis accuracy. However, AI systems require large, clean datasets and careful validation before clinical use. Privacy, bias, and accountability are major concerns. If used responsibly, AI can support clinicians rather than replace them, making healthcare faster, safer, and more equitable for diverse populations.""",

    """Climate change is driven by greenhouse gas emissions from burning fossil fuels. Renewable energy sources such as solar, wind, and hydropower can reduce carbon pollution. Governments and businesses are investing in batteries, smart grids, and energy efficiency. Communities also need adaptation plans for heat waves, floods, and droughts. Climate policy involves trade-offs between economic growth and environmental protection. Individual actions matter, but large-scale systemic change is essential. Accurate data and international cooperation help track progress toward lowering emissions and limiting global temperature rise.""",

    """Basketball strategy depends on spacing, passing, and player movement. Coaches design plays to create open shots and mismatches. Offenses use pick-and-rolls, screens, and fast breaks, while defenses switch, trap, or zone. Analytics now influences decisions about shot selection, lineup combinations, and pace. A team with strong chemistry often outperforms a group of individual stars. Rebounding and turnovers frequently decide close games. Effective communication on defense is as important as scoring ability. Successful teams practice repeatedly and adjust tactics based on opponent weaknesses and game situations.""",

    """Italian pasta cooking begins with quality ingredients and simple techniques. A good sauce balances tomatoes, olive oil, garlic, herbs, and sometimes meat or cheese. Pasta should be cooked in salted water until al dente, then finished in the sauce for better flavor. Fresh basil and parmesan add aroma and richness. Different shapes suit different sauces: long pasta for light sauces, short tubes for chunky sauces. Regional traditions vary widely across Italy. Home cooks can improve by tasting, seasoning gradually, and respecting the pasta's texture rather than relying only on a timer.""",

    """Space exploration has advanced through robotic missions and human spaceflight. Mars rovers study geology, atmosphere, and signs of ancient water. Telescopes observe distant galaxies, exoplanets, and black holes. Space agencies and private companies are developing reusable rockets to lower launch costs. Astronauts face radiation, isolation, and microgravity health risks. Future missions may establish lunar bases or send crews to Mars. International collaboration and data sharing accelerate discovery. Exploring space also produces technologies that benefit life on Earth, from satellite navigation to medical devices and materials science.""",

    """Cybersecurity protects computers, networks, and personal data from attacks. Common threats include phishing emails, malware, ransomware, and weak passwords. Multi-factor authentication and regular software updates reduce risk. Organizations use firewalls, encryption, and monitoring systems to detect intrusions. Employees are often the weakest link, so security training is essential. Data breaches can damage finances and reputation. Strong cybersecurity requires layered defenses, incident response plans, and privacy laws. As attackers become more sophisticated, defenders must continuously learn new tools, share threat intelligence, and design systems that are secure by default.""",

    """Music theory explains scales, chords, rhythm, and harmony. Musicians practice intervals, chord progressions, and ear training to perform and compose. Different genres use distinct rhythms and instrumentation. A melody becomes memorable through repetition, contrast, and tension. Improvisation requires both theoretical knowledge and listening skills. Recording software allows artists to edit, mix, and produce tracks at home. Live performance depends on timing, dynamics, and communication with bandmates. Learning music improves memory, coordination, and emotional expression. Consistent practice, feedback, and experimentation help musicians develop a personal style over time.""",

    """Gardening vegetables requires healthy soil, sunlight, water, and patience. Compost adds nutrients and improves soil structure. Gardeners rotate crops to prevent pests and diseases. Tomatoes, peppers, beans, and lettuce grow well in many home gardens. Mulch conserves moisture and reduces weeds. Raised beds help control drainage and soil quality. Pests can be managed with beneficial insects, companion planting, and careful monitoring. Harvesting at the right time improves flavor and nutrition. A successful garden depends on observing local climate, planning seasons, and learning from each growing cycle year after year.""",

    """Economics studies how people, firms, and governments allocate scarce resources. Inflation measures the rising cost of goods and services over time. Central banks adjust interest rates to influence borrowing, spending, and investment. High inflation reduces purchasing power, especially for low-income households. Unemployment and economic growth are key indicators of national health. Fiscal policy involves taxation and government spending. Global trade, supply chains, and energy prices affect local economies. Understanding incentives, markets, and data helps policymakers design stable policies. Economic decisions always involve trade-offs between efficiency, equity, and long-term sustainability.""",

    """Public health focuses on preventing disease and improving community well-being. Sleep, exercise, and nutrition strongly influence physical and mental health. Regular activity strengthens the heart, muscles, and immune system. A balanced diet includes vegetables, fruits, whole grains, and protein. Chronic stress can harm sleep, mood, and metabolism. Vaccination, clean water, and health education reduce illness. Public health campaigns encourage healthy habits and early screening. Access to care and social support also shape outcomes. Small daily choices, supportive environments, and evidence-based policies can help populations live longer, healthier, and more productive lives."""
]


def load_documents():
    """Return (titles, documents)."""
    return TITLES, DOCUMENTS


titles, documents = load_documents()
print(f"Loaded {len(documents)} documents.")
for t, d in zip(titles, documents):
    print(f"  - {t:<22} {len(d.split()):>3} words")

Loaded 10 documents.
  - AI in Healthcare        85 words
  - Climate Change          83 words
  - Basketball Strategy     85 words
  - Italian Pasta           91 words
  - Space Exploration       86 words
  - Cybersecurity           88 words
  - Music Theory            87 words
  - Gardening Vegetables    89 words
  - Economics Inflation     89 words
  - Public Health           91 words


### NLTK preprocessing
Tokenize → lowercase → remove NLTK English stopwords → lemmatize (`emissions` → `emission`).

In [3]:
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer

STOP_WORDS = set(stopwords.words("english"))
TOKENIZER = RegexpTokenizer(r"[a-zA-Z]+(?:-[a-zA-Z]+)*")
LEMMATIZER = WordNetLemmatizer()


def nltk_preprocess(text):
    tokens = TOKENIZER.tokenize(text.lower())
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return [t for t in tokens if t not in STOP_WORDS]


print("NLTK stopword list size:", len(STOP_WORDS))
sample = documents[0].split(".")[0]
print("raw   :", sample)
print("tokens:", nltk_preprocess(sample))

NLTK stopword list size: 198
raw   : Artificial intelligence is transforming healthcare by helping doctors analyze medical images, predict disease risk, and personalize treatments
tokens: ['artificial', 'intelligence', 'transforming', 'healthcare', 'helping', 'doctor', 'analyze', 'medical', 'image', 'predict', 'disease', 'risk', 'personalize', 'treatment']


## 2. Build the TF-IDF matrix

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer


def build_tfidf(documents, ngram_range=(1, 1), max_df=1.0, min_df=1):
    """Fit TfidfVectorizer on NLTK-preprocessed tokens (English stopwords removed by NLTK)."""
    vectorizer = TfidfVectorizer(
        tokenizer=nltk_preprocess,
        lowercase=False,
        token_pattern=None,
        ngram_range=ngram_range,
        max_df=max_df,
        min_df=min_df,
    )
    return vectorizer, vectorizer.fit_transform(documents)


vectorizer, X = build_tfidf(documents)
vocab = vectorizer.get_feature_names_out()
print("Matrix shape   :", X.shape)        # (10, n_unique_terms)
print("Vocabulary size:", len(vocab))

Matrix shape   : (10, 527)
Vocabulary size: 527


## 3. Inspect the matrix (NumPy, no DataFrame)

In [5]:
def print_matrix_sample(matrix, vectorizer, titles, n_docs=3, n_terms=10):
    terms = vectorizer.get_feature_names_out()[:n_terms]
    dense = matrix[:n_docs, :n_terms].toarray()
    w = max(len(t) for t in terms) + 2
    print(" " * 22 + "".join(f"{t:>{w}}" for t in terms))
    for title, row in zip(titles[:n_docs], dense):
        print(f"{title[:20]:<22}" + "".join(f"{v:>{w}.3f}" for v in row))


def top_terms_per_document(matrix, vectorizer, titles, n=5):
    terms = vectorizer.get_feature_names_out()
    result = {}
    for i, title in enumerate(titles):
        row = matrix[i].toarray().ravel()
        idx = np.argsort(row)[::-1][:n]
        result[title] = [(terms[j], float(row[j])) for j in idx if row[j] > 0]
    return result


print("Sample (first 3 docs x 10 terms):")
print_matrix_sample(X, vectorizer, titles)

print("\nTop 5 TF-IDF terms per document:")
for title, terms in top_terms_per_document(X, vectorizer, titles).items():
    print(f"  {title:<22} " + ", ".join(f"{t}({s:.2f})" for t, s in terms))

Sample (first 3 docs x 10 terms):
                               ability      accelerate          access  accountability        accuracy        accurate          across          action        activity      adaptation
AI in Healthcare                 0.000           0.000           0.000           0.122           0.122           0.000           0.000           0.000           0.000           0.000
Climate Change                   0.000           0.000           0.000           0.000           0.000           0.125           0.000           0.125           0.000           0.125
Basketball Strategy              0.123           0.000           0.000           0.000           0.000           0.000           0.000           0.000           0.000           0.000

Top 5 TF-IDF terms per document:
  AI in Healthcare       ai(0.37), healthcare(0.24), use(0.16), bias(0.12), transforming(0.12)
  Climate Change         change(0.25), emission(0.25), climate(0.21), energy(0.21), business(0.13)
  Bask

## 4. Query similarity challenge

In [6]:
from sklearn.metrics.pairwise import cosine_similarity


def rank_documents_for_query(query, vectorizer, matrix, titles):
    """Return [(title, score)] sorted by cosine similarity, highest first."""
    q_vec = vectorizer.transform([query])          # same NLTK pipeline
    scores = cosine_similarity(q_vec, matrix).ravel()
    return sorted(zip(titles, scores.tolist()), key=lambda x: x[1], reverse=True)


QUERIES = {
    "renewable energy and climate policy": "Climate Change",
    "password security and phishing": "Cybersecurity",
    "healthy sleep exercise and diet": "Public Health",
}

for q, expected in QUERIES.items():
    ranking = rank_documents_for_query(q, vectorizer, X, titles)
    print(f"\nQuery: {q!r}  -> tokens {nltk_preprocess(q)}")
    for r, (title, score) in enumerate(ranking, 1):
        print(f"  {r:>2}. {title:<22} {score:.3f}")
    ok = "OK" if ranking[0][0] == expected else "MISMATCH"
    print(f"  top: {ranking[0][0]} (expected {expected}) [{ok}]")


Query: 'renewable energy and climate policy'  -> tokens ['renewable', 'energy', 'climate', 'policy']
   1. Climate Change         0.322
   2. Economics Inflation    0.125
   3. Gardening Vegetables   0.049
   4. Public Health          0.037
   5. AI in Healthcare       0.000
   6. Basketball Strategy    0.000
   7. Italian Pasta          0.000
   8. Space Exploration      0.000
   9. Cybersecurity          0.000
  10. Music Theory           0.000
  top: Climate Change (expected Climate Change) [OK]

Query: 'password security and phishing'  -> tokens ['password', 'security', 'phishing']
   1. Cybersecurity          0.211
   2. AI in Healthcare       0.000
   3. Climate Change         0.000
   4. Basketball Strategy    0.000
   5. Italian Pasta          0.000
   6. Space Exploration      0.000
   7. Music Theory           0.000
   8. Gardening Vegetables   0.000
   9. Economics Inflation    0.000
  10. Public Health          0.000
  top: Cybersecurity (expected Cybersecurity) [OK]

Quer

## 5. Optional challenge — bigrams, `max_df=0.8`, `min_df=2`

In [7]:
v2, X2 = build_tfidf(documents, ngram_range=(1, 2), max_df=0.8, min_df=2)
print("Matrix shape   :", X2.shape)
print(f"Vocabulary size: {len(v2.get_feature_names_out())} (was {len(vocab)})")
print("Bigrams kept   :", [t for t in v2.get_feature_names_out() if " " in t])
print("\nTop 5 terms per document:")
for title, terms in top_terms_per_document(X2, v2, titles).items():
    print(f"  {title:<22} " + (", ".join(f"{t}({s:.2f})" for t, s in terms) or "(none)"))

Matrix shape   : (10, 76)
Vocabulary size: 76 (was 527)
Bigrams kept   : ['economic growth', 'policy involves']

Top 5 terms per document:
  AI in Healthcare       use(0.38), support(0.24), population(0.24), careful(0.24), improve(0.24)
  Climate Change         energy(0.39), climate(0.39), track(0.20), trade-off(0.20), plan(0.20)
  Basketball Strategy    defense(0.52), successful(0.26), strong(0.26), individual(0.26), often(0.26)
  Italian Pasta          different(0.55), tomato(0.27), shape(0.27), quality(0.27), add(0.27)
  Space Exploration      study(0.31), life(0.31), international(0.31), produce(0.31), medical(0.31)
  Cybersecurity          system(0.38), data(0.34), defense(0.22), training(0.22), privacy(0.22)
  Music Theory           practice(0.47), training(0.24), software(0.24), track(0.24), communication(0.24)
  Gardening Vegetables   improves(0.42), vegetable(0.21), successful(0.21), tomato(0.21), healthy(0.21)
  Economics Inflation    government(0.36), economic(0.36), policy(

**Observation:** `min_df=2` keeps only terms found in 2+ documents, so the vocabulary collapses and topic-specific words disappear. Only a couple of bigrams survive, and top terms become *less* distinctive (e.g. "different", "often"). On a 10-document corpus, bigrams don't make terms more meaningful.

## 6. Verify — the unit tests from `tests/test_tfidf_utils.py`, run with pytest

In [8]:
!pip install -q pytest ipytest
import ipytest
ipytest.autoconfig()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.0 MB/s eta 0:00:00


In [9]:
%%ipytest -q

def test_load_documents():
    titles, docs = load_documents()
    assert len(titles) == 10
    assert len(docs) == 10
    assert all(len(d.split()) > 30 for d in docs)  # "long text"


def test_tfidf_matrix_shape():
    _, docs = load_documents()
    vectorizer, matrix = build_tfidf(docs)
    assert matrix.shape[0] == 10
    assert matrix.shape[1] == len(vectorizer.get_feature_names_out())


def test_query_ranking_matches_expected_topic():
    titles, docs = load_documents()
    vectorizer, matrix = build_tfidf(docs)
    ranking = rank_documents_for_query(
        "renewable energy and climate policy", vectorizer, matrix, titles
    )
    assert ranking[0][0] == "Climate Change"

...                                                                                          [100%]
